# BICE RAG — Notebook de Evaluación (RAGAS)
Mide la calidad del sistema RAG con métricas reales.

**Métricas que vamos a medir:**
- **Faithfulness**: ¿la respuesta está basada en el contexto recuperado? (evita alucinaciones)
- **Answer Relevancy**: ¿la respuesta responde la pregunta?
- **Context Precision**: ¿los chunks recuperados son relevantes para la pregunta?
- **Context Recall**: ¿se recuperó suficiente información para responder bien?

## 0. Setup

In [ ]:
%pip install ragas datasets pandas matplotlib seaborn --quiet

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset

sys.path.insert(0, '../backend')

print('✅ Setup completo')

## 1. Dataset de evaluación
Preguntas con respuestas de referencia (ground truth) para medir contra el sistema.

Estas preguntas están diseñadas para cubrir casos positivos, negativos y edge cases.

In [ ]:
# Dataset de evaluación — pregunta + respuesta esperada (ground truth)
# Basado en los documentos reales de backend/data/docs/

EVAL_DATASET = [
    # --- SEGURO AUTO ---
    {
        'question': '¿Qué cubre el seguro de auto en caso de robo total?',
        'ground_truth': 'El seguro cubre el robo total del vehículo. El siniestro debe ser denunciado a Carabineros o PDI dentro de las 24 horas. El monto máximo es el valor comercial del vehículo con un deducible de 5 UF.',
        'tipo_seguro': 'auto'
    },
    {
        'question': '¿Está cubierto un conductor que manejaba bajo el efecto del alcohol?',
        'ground_truth': 'No. Los daños causados mientras el conductor esté bajo la influencia del alcohol con alcoholemia superior a 0,3 g/l están excluidos de la cobertura.',
        'tipo_seguro': 'auto'
    },
    {
        'question': '¿Cuánto tiempo tengo para denunciar un siniestro de auto?',
        'ground_truth': 'Debes llamar al número de siniestros BICE 600 600 2424 dentro de las 72 horas del evento.',
        'tipo_seguro': 'auto'
    },
    {
        'question': '¿Cubre el seguro de auto si uso el vehículo como Uber?',
        'ground_truth': 'No. Los vehículos usados como aplicación de transporte como Uber o Cabify sin declaración previa están excluidos de la cobertura.',
        'tipo_seguro': 'auto'
    },
    # --- SEGURO VIDA ---
    {
        'question': '¿Qué es la invalidez total y permanente en el seguro de vida?',
        'ground_truth': 'Es la pérdida total e irreversible de la capacidad de trabajar en cualquier actividad remunerada. Requiere certificación de dos especialistas. Tiene un período de espera de 6 meses para enfermedades preexistentes.',
        'tipo_seguro': 'vida'
    },
    {
        'question': '¿Cubre el seguro de vida el suicidio?',
        'ground_truth': 'No durante los primeros 2 años de vigencia de la póliza. El suicidio en los primeros 2 años está excluido.',
        'tipo_seguro': 'vida'
    },
    {
        'question': '¿Hasta qué edad se puede renovar el seguro de vida?',
        'ground_truth': 'La edad máxima de renovación es 70 años.',
        'tipo_seguro': 'vida'
    },
    # --- SEGURO HOGAR ---
    {
        'question': '¿Qué diferencia hay entre el plan propietario y el plan arrendatario?',
        'ground_truth': 'El plan Propietario cubre estructura del inmueble más contenido. El plan Arrendatario cubre solo el contenido como muebles y electrodomésticos.',
        'tipo_seguro': 'hogar'
    },
    {
        'question': '¿Cubre el seguro de hogar el robo sin evidencia de fuerza?',
        'ground_truth': 'No. El seguro cubre robo con fuerza cuando existe evidencia como cerraduras forzadas o vidrios rotos. El robo sin evidencia de fuerza está excluido.',
        'tipo_seguro': 'hogar'
    },
    # --- FAQ GENERAL ---
    {
        'question': '¿Qué pasa si no pago la prima del seguro de auto?',
        'ground_truth': 'La póliza se suspende a los 10 días del vencimiento sin pago. BICE enviará aviso de suspensión por correo electrónico y carta certificada.',
        'tipo_seguro': None
    },
    # --- RETRIEVAL GUARD (debe decir no encontrado) ---
    {
        'question': '¿Cuánto vale el dólar hoy?',
        'ground_truth': 'No encontré información sobre eso en los documentos disponibles.',
        'tipo_seguro': None
    },
    {
        'question': '¿Cómo está el mercado bursátil?',
        'ground_truth': 'No encontré información sobre eso en los documentos disponibles.',
        'tipo_seguro': None
    },
]

print(f'Dataset de evaluación: {len(EVAL_DATASET)} preguntas')
print(f'  - Auto: {sum(1 for x in EVAL_DATASET if x["tipo_seguro"]=="auto")}')
print(f'  - Vida: {sum(1 for x in EVAL_DATASET if x["tipo_seguro"]=="vida")}')
print(f'  - Hogar: {sum(1 for x in EVAL_DATASET if x["tipo_seguro"]=="hogar")}')
print(f'  - General/Guard: {sum(1 for x in EVAL_DATASET if x["tipo_seguro"] is None)}')

## 2. Generar respuestas del sistema
Correr cada pregunta contra el RAG y guardar respuestas + contextos.

In [ ]:
from src.chain.rag_chain import ask
from src.retrieval.retriever import get_relevant_chunks

results = []

for i, item in enumerate(EVAL_DATASET):
    print(f'[{i+1}/{len(EVAL_DATASET)}] {item["question"][:60]}...')
    
    # Obtener chunks recuperados (para context precision/recall)
    chunks = get_relevant_chunks(item['question'], item['tipo_seguro'])
    contexts = [c.page_content for c in chunks] if chunks else []
    
    # Obtener respuesta del sistema
    response = ask(item['question'], session_id=f'eval-{i}', tipo_seguro=item['tipo_seguro'])
    
    results.append({
        'question': item['question'],
        'answer': response['answer'],
        'contexts': contexts,
        'ground_truth': item['ground_truth'],
        'chunks_used': response['chunks_used'],
        'sources': response['sources'],
        'tipo_seguro': item['tipo_seguro']
    })

print(f'\n✅ {len(results)} respuestas generadas')

## 3. Evaluación con RAGAS

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Configurar RAGAS para usar Ollama local (sin OpenAI)
ragas_llm = LangchainLLMWrapper(ChatOllama(model='llama3.2', temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(OllamaEmbeddings(model='nomic-embed-text'))

# Preparar dataset para RAGAS
# Filtrar solo los que tienen contexto (excluir retrieval guard cases)
eval_with_context = [r for r in results if len(r['contexts']) > 0]

ragas_dataset = Dataset.from_dict({
    'question': [r['question'] for r in eval_with_context],
    'answer': [r['answer'] for r in eval_with_context],
    'contexts': [r['contexts'] for r in eval_with_context],
    'ground_truth': [r['ground_truth'] for r in eval_with_context],
})

print(f'Evaluando {len(eval_with_context)} preguntas con contexto...')
print('(Excluidas las preguntas de retrieval guard sin contexto)')

In [ ]:
# Correr evaluación RAGAS — puede tardar varios minutos
ragas_results = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

print('\n✅ Evaluación completa')
print('\nScores globales:')
print(f'  Faithfulness:      {ragas_results["faithfulness"]:.3f}  (evita alucinaciones — más alto mejor)')
print(f'  Answer Relevancy:  {ragas_results["answer_relevancy"]:.3f}  (responde la pregunta — más alto mejor)')
print(f'  Context Precision: {ragas_results["context_precision"]:.3f}  (chunks relevantes — más alto mejor)')
print(f'  Context Recall:    {ragas_results["context_recall"]:.3f}  (info suficiente recuperada — más alto mejor)')

## 4. Visualización de resultados

In [ ]:
# Scores por pregunta
df_scores = ragas_results.to_pandas()
df_scores['question_short'] = df_scores['question'].str[:50] + '...'

display(df_scores[['question_short', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']])

In [ ]:
# Gráfico de radar de métricas globales
import numpy as np

metrics = ['Faithfulness', 'Answer\nRelevancy', 'Context\nPrecision', 'Context\nRecall']
values = [
    ragas_results['faithfulness'],
    ragas_results['answer_relevancy'],
    ragas_results['context_precision'],
    ragas_results['context_recall']
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#2ecc71' if v >= 0.7 else '#f39c12' if v >= 0.5 else '#e74c3c' for v in values]
bars = ax1.bar(metrics, values, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_ylim(0, 1)
ax1.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='Umbral aceptable (0.7)')
ax1.set_title('BICE RAG — Métricas RAGAS', fontsize=14, fontweight='bold')
ax1.set_ylabel('Score (0-1)')
ax1.legend()
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', fontweight='bold')

# Heatmap por pregunta
heatmap_data = df_scores[['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']]
sns.heatmap(heatmap_data, ax=ax2, cmap='RdYlGn', vmin=0, vmax=1,
            annot=True, fmt='.2f', cbar_kws={'label': 'Score'},
            yticklabels=[f'Q{i+1}' for i in range(len(heatmap_data))])
ax2.set_title('Score por pregunta', fontsize=14, fontweight='bold')
ax2.set_xticklabels(['Faith.', 'Ans.Rel.', 'Ctx.Prec.', 'Ctx.Rec.'], rotation=30)

plt.tight_layout()
plt.savefig('ragas_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado como ragas_results.png')

## 5. Evaluación del Retrieval Guard
Verifica que las preguntas fuera de dominio sean bloqueadas correctamente.

In [ ]:
# Evaluar casos de retrieval guard separadamente
guard_cases = [r for r in results if r['chunks_used'] == 0]
expected_fallback = 'No encontré información sobre eso en los documentos disponibles.'

print(f'Casos de retrieval guard: {len(guard_cases)}')
print('-' * 60)

guard_correct = 0
for case in guard_cases:
    correct = expected_fallback in case['answer']
    status = '✅ CORRECTO' if correct else '❌ FALLÓ'
    if correct:
        guard_correct += 1
    print(f'{status} | "{case["question"][:50]}"')
    if not correct:
        print(f'         Respuesta: {case["answer"][:100]}')

print(f'\nRetrieval Guard Accuracy: {guard_correct}/{len(guard_cases)} ({guard_correct/len(guard_cases)*100:.0f}%)')

## 6. Resumen ejecutivo
Interpretación de resultados para la entrevista.

In [ ]:
print('=' * 60)
print('RESUMEN EJECUTIVO — BICE RAG MVP')
print('=' * 60)

scores = {
    'Faithfulness': ragas_results['faithfulness'],
    'Answer Relevancy': ragas_results['answer_relevancy'],
    'Context Precision': ragas_results['context_precision'],
    'Context Recall': ragas_results['context_recall'],
}

for metric, score in scores.items():
    if score >= 0.7:
        status = '✅ ACEPTABLE'
    elif score >= 0.5:
        status = '⚠️  MEJORABLE'
    else:
        status = '❌ CRÍTICO'
    print(f'  {metric:<22} {score:.3f}  {status}')

print()
avg = sum(scores.values()) / len(scores)
print(f'  Score promedio: {avg:.3f}')
print()
print('Interpretación para entrevista:')
print('  - Faithfulness alta = el sistema no alucina')
print('  - Context Precision alta = el reranker funciona bien')
print('  - Si Context Recall es baja = chunk_size o k deben ajustarse')
print('=' * 60)